# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()
print(f"{metadata_json['name']}: {metadata_json['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll examine the dataset's record set structure, referencing all entities by their `@id`.

First, let's list all available record sets. Then, for each, inspect the fields and columns (all referenced by their `@id`).

In [ ]:
# Display available recordSets in the dataset by @id
record_sets = dataset.record_sets
print("Record Sets (by @id):")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', 'Unnamed')}")

# Pick the first record set for further inspection
first_record_set_id = record_sets[0]['@id'] if len(record_sets) > 0 else None
if first_record_set_id:
    print(f"\nFields for record set: {first_record_set_id}")
    fields = dataset.fields(record_set=first_record_set_id)
    for field in fields:
        print(f"- {field['@id']}: {field.get('name', 'Unnamed')} (dataType: {field.get('dataType', 'Unknown')})")

# Optionally, print column info if present
    print("\nColumns (by @id):")
    columns = dataset.columns(record_set=first_record_set_id)
    for col in columns:
        print(f"- {col['@id']}: {col.get('name', 'Unnamed')} (dataType: {col.get('dataType', 'Unknown')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview above.

We'll extract records from each record set, referencing each by its `@id`, and load into pandas DataFrames.

In [ ]:
# Extract data from all listed record sets by their @id
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Display column names for the first record set's dataframe
if first_record_set_id in dataframes:
    print(f"Columns for record set {first_record_set_id}:")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

- Removing outliers
- Transforming data distributions
- Grouping by key demographic or survey attributes

Use field/column `@id` for dynamic referencing.

In [ ]:
from IPython.display import display
import numpy as np

# Example EDA: Filter and normalize a numeric survey score (e.g., GAD-7)
# We'll find a numeric field in the record set

df = dataframes[first_record_set_id]

# Find a numeric field: heuristically look for column with 'score' or known numeric
possible_numeric_fields = [col for col in df.columns if 'score' in col.lower() or col.lower() in ['age', 'gad_7', 'phq_9', 'psq']]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]  # Use @id (column name)
    threshold = df[numeric_field_id].mean() if np.issubdtype(df[numeric_field_id].dtype, np.number) else 10

    # Filter
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a demographic field, e.g., 'gender' or 'age_group'
    possible_group_fields = [col for col in df.columns if col.lower() in ['gender', 'marital_status', 'village', 'age_group']]
    if possible_group_fields:
        group_field_id = possible_group_fields[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df.head())
else:
    print("No suitable numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the numeric survey score field selected previously, and visualize its relationship with a demographic variable (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of numeric field
if possible_numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Visualize by group (if available)
    if possible_group_fields:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded dataset metadata and structure using `mlcroissant`, referencing all entities by their `@id`.
- Inspected record sets, fields, and columns.
- Extracted the main survey record set and explored numeric fields such as GAD-7, PHQ-9, or PSQ scores.
- Applied filtering, normalization, and grouped analysis by demographic fields (e.g., gender, village).
- Visualized key distributions and relationships.

This FAIR² mental health survey dataset enables informed analysis of psychological and demographic trends in Kilifi County, Kenya, and demonstrates AI-ready, interoperable data standards for reproducible research.